In [15]:
import pandas as pd
from statsforecast.models import MSTL
from google.cloud import bigquery
import datetime
import multiprocessing
from tqdm import tqdm
import yaml

In [16]:
with open("config.yaml", "r") as stream:
    try:
        anomaly_config = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

In [17]:
forecast_config = anomaly_config["anomalies"]["brazil_vms_production_v20211126"]

In [18]:
forecast_config

{'source_dataset': 'pipe_brazil_production_v20211126',
 'source_table': 'brazil_vms_normalized_20231023',
 'source_date_column': 'date',
 'source_date_column_sql': 'date(timestamp)',
 'source_forecast_column': 'count',
 'source_forecast_column_sql': 'COUNT(*)',
 'source_sql': ' SELECT PARSE_DATE("%Y%m%d", REGEXP_REPLACE(table_id, "brazil_vms_normalized_(.*)", "\\\\1")) date, row_count y FROM `world-fishing-827.pipe_brazil_production_v20211126.__TABLES__` WHERE table_id LIKE "brazil_vms_normalized_%" AND PARSE_DATE("%Y%m%d", REGEXP_REPLACE(table_id, "brazil_vms_normalized_(.*)", "\\\\1")) BETWEEN "1979-01-01" AND "2099-01-01" ORDER BY PARSE_DATE("%Y%m%d", REGEXP_REPLACE(table_id, "brazil_vms_normalized_(.*)", "\\\\1")) ',
 'algorithms': {'mstl': {'parameters': {'season_length': [365, 7]},
   'train_start': 'Inf',
   'train_end': 'Inf',
   'forecast_periods': 1}},
 'target_dataset': 'scratch_christian_homberg_ttl120d',
 'target_table': 'anomaly_detection_forecasts'}

In [19]:
client = bigquery.Client()

/mnt/encrypted_data/git/data-testing/venv/lib/python3.9/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/mnt/encrypted_data/git/data-testing/venv/lib/python3.9/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [80]:
import pandas as pd
from google.cloud import bigquery
import hashlib
import os
import pyarrow as pa
import pyarrow.feather as feather
from google.cloud.bigquery import QueryJobConfig
import warnings

BQ_KB = 1024
BQ_MB = BQ_KB * 1024
BQ_GB = BQ_MB * 1024
allowed_size = 0.1 * BQ_GB

def estimate_query_size(query, billing='world-fishing-827'):
    client = bigquery.Client(billing)
    job_config = QueryJobConfig(dry_run=True, use_query_cache=False)
    query_job = client.query(query, job_config=job_config)
    return query_job.total_bytes_processed

def validate_query_size(query, allowed_size=0.1 * BQ_GB):
    query_estimate = estimate_query_size(query)
    if query_estimate > allowed_size:
        warnings.warn(f"Query exceeds allowed_size "
                      f"allowed_size = {allowed_size / BQ_GB} GB, "
                      f"estimated size = {query_estimate / BQ_GB} GB")
    elif query_estimate * 1.5 < allowed_size:
        warnings.warn(f"Query allowed_size is set more than 50% higher than estimated_size. "
                      f"You can and probably should set the allowed_size closer to the estimated_size "
                      f"allowed_size = {allowed_size / BQ_GB} GB, "
                      f"estimated size = {query_estimate / BQ_GB} GB")
    return query_estimate <= allowed_size

def safe_query(query, project_id, allowed_size=0.1 * BQ_GB, 
               query_size_exceeded_handling='stop', page_size=None):
    if validate_query_size(query, allowed_size):
        df = pd.read_gbq(query, project_id=project_id, progress_bar_type=None)
        return df
    else:
        if query_size_exceeded_handling == 'stop':
            raise Exception("Query size limit exceeded!")
        elif query_size_exceeded_handling == 'skip':
            warnings.warn("Query size limit exceeded!")

def safe_cached_query(query, project_id=None, allowed_size=None,
                      cache_dir=".cached_queries", cache_version=None, 
                      overwrite_if_cached=False, verbose=False, 
                      query_size_exceeded_handling='stop', page_size=None, silent=False):
    # Format the SQL query
    formatted_query = sqlparse.format(query, reindent=True, keyword_case='upper')

    # Print the formatted SQL query if verbose is True
    if verbose:
        print(formatted_query)

    # Hash the formatted query
    query_hash = hashlib.md5(formatted_query.encode()).hexdigest()

    # Determine the cache directory
    if cache_version is not None:
        query_dir = os.path.join(cache_dir, cache_version)
    else:
        query_dir = cache_dir

    # Set allowed_size if it is not provided
    if allowed_size is None:
        allowed_size = 0.1 * BQ_GB

    # Define file paths
    query_path = os.path.join(query_dir, query_hash)
    query_sql_path = f"{query_path}.sql"

    # Check if the query is cached
    if not overwrite_if_cached and os.path.exists(query_path):
        if not silent:
            print("Query found in cache - retrieving result")
        return feather.read_feather(query_path)

    # If not cached or overwrite is requested, run the query
    df = safe_query(formatted_query, project_id, allowed_size, query_size_exceeded_handling, page_size)

    # Cache the result
    if df is not None:
        if not os.path.exists(query_dir):
            os.makedirs(query_dir)
        feather.write_feather(df, query_path)
        with open(query_sql_path, "w") as f:
            f.write(formatted_query)

    return df


In [81]:
def get_training_data(
        source_sql=None, 
        source_dataset=None, 
        source_table=None, 
        source_date_column_sql=None, 
        source_forecast_column_sql=None, 
        train_start=None, 
        train_end=None,
        **kwargs
):
    if (source_sql is not None):
        source_sql = source_sql.format(source_date_column_sql=source_date_column_sql, train_start=train_start, train_end=train_end)
        df_timeseries = safe_cached_query(source_sql, silent=True)
    else:
        df_timeseries = pd.read_gbq(f'''
        SELECT 
            {source_date_column_sql} date, 
            {source_forecast_column_sql} y
        FROM {source_dataset}.{source_table}
        WHERE {source_date_column_sql} BETWEEN '{train_start}' AND '{train_end}'
        GROUP BY date
        ORDER BY date
        ''')

    df_timeseries["date"]=pd.to_datetime(df_timeseries["date"])
    df_timeseries = df_timeseries.query('date >= @train_start and date <= @train_end')

    return(df_timeseries)

In [82]:
def get_mstl_forecast(forecast_config):
    forecast_config_copy = dict(forecast_config)
    FORECAST_DATE = forecast_config_copy["FORECAST_DATE"]
    mstl_config = forecast_config_copy["algorithms"]["mstl"]
    if mstl_config["train_start"] == "Inf":
        train_start="1979-01-01"
    else:
        train_start=mstl_config["train_start"]
        
    if mstl_config["train_end"] == "Inf":
        train_end=(datetime.date.fromisoformat(FORECAST_DATE) - datetime.timedelta(days=1)).isoformat()
    else:
        train_end=mstl_config["train_end"]

    df_training_data=get_training_data(
        **forecast_config_copy,
        train_start=train_start,
        train_end=train_end
    )
    np_mstl_train=df_training_data["y"].to_numpy().astype(int)
    mstl_model = MSTL(**mstl_config["parameters"])
    mstl_forecasts = mstl_model.forecast(np_mstl_train, mstl_config["forecast_periods"])["mean"]
    mstl_train_end = datetime.date.fromisoformat(train_end)
    mstl_forecast_dates = [(mstl_train_end + datetime.timedelta(days=d)) for d in range(1, mstl_config["forecast_periods"] + 1)]
    mstl_train_dates = pd.date_range(min(df_training_data["date"]), max(df_training_data["date"]))
    forecast_config_copy["forecast_algorithm"] = "mstl"
    forecast_config_copy["forecasts"] = [{"date": k, "value": v} for k,v in zip(mstl_forecast_dates, mstl_forecasts)]
    forecast_config_copy["actuals"] = [{"date": k, "value": v} for k,v in zip(mstl_train_dates, np_mstl_train)]
    return(forecast_config_copy)

In [83]:
forecast_configs = []
forecast_list = []
for date_offset in range(1, 1390):
    current_forecast_date = (datetime.date.fromisoformat("2020-01-01") + datetime.timedelta(days=date_offset)).isoformat()
    current_forecast_config = dict(forecast_config)
    current_forecast_config["FORECAST_DATE"] = current_forecast_date
    forecast_configs.append(current_forecast_config)

In [84]:
pbar.close()
pbar = tqdm(total=len(forecast_configs))
with multiprocessing.Pool(12) as pool:
    for forecast in pool.imap_unordered(get_mstl_forecast, forecast_configs):
        forecast_list.append(forecast)
        pbar.update(1)

pbar.close()

  0%|          | 0/1389 [00:00<?, ?it/s]/mnt/encrypted_data/git/data-testing/venv/lib/python3.9/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/mnt/encrypted_data/git/data-testing/venv/lib/python3.9/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/mnt/encrypted_data/git/data-testing/venv/lib/pyth

Query found in cache - retrieving result
Query found in cache - retrieving result
Query found in cache - retrieving result


  0%|          | 3/1389 [00:45<4:35:33, 11.93s/it] 

Query found in cache - retrieving result
Query found in cache - retrieving result


  0%|          | 5/1389 [00:46<2:20:23,  6.09s/it]

...

 99%|█████████▉| 1374/1389 [04:39<00:03,  4.67it/s]

Query found in cache - retrieving result
Query found in cache - retrieving result


 99%|█████████▉| 1377/1389 [04:39<00:02,  4.55it/s]

Query found in cache - retrieving result


100%|██████████| 1389/1389 [04:41<00:00,  4.93it/s]


In [85]:
df_forecasts = pd.DataFrame.from_dict(forecast_list) \
    .drop(["FORECAST_DATE", "target_dataset", "target_table"], axis=1) \
    .assign(execution_time = datetime.datetime.now())

df_forecasts = df_forecasts.astype({"algorithms": "string"})

In [86]:
forecasts_schema = [
    {"name": "source_dataset", "type": "STRING"},
    {"name": "source_table", "type": "STRING"},
    {"name": "source_date_column", "type": "STRING"},
    {"name": "source_date_column_sql", "type": "STRING"},
    {"name": "source_forecast_column", "type": "STRING"},
    {"name": "source_forecast_column_sql", "type": "STRING"},
    {"name": "source_sql", "type": "STRING"},
    {"name": "algorithms", "type": "STRING"},
    {"name": "algorithm", "type": "STRING"},
    {"name": "forecast_algorithm", "type": "STRING"},
    {
        "name": "forecasts", "type": "RECORD", "mode": "REPEATED", "fields": [
            {"name": "date", "type": "DATE"}, 
            {"name": "value", "type": "FLOAT"}
        ]
    },
    {
        "name": "actuals", "type": "RECORD", "mode": "REPEATED", "fields": [
            {"name": "date", "type": "DATE"}, 
            {"name": "value", "type": "FLOAT"}
        ]
    },
    {"name": "execution_time", "type": "DATETIME"}
]

target_dataset = forecast_configs[0]["target_dataset"]
target_table = forecast_configs[0]["target_table"]
df_forecasts.to_gbq(destination_table=f"{target_dataset}.{target_table}", if_exists="append", table_schema=forecasts_schema)

100%|██████████| 1/1 [00:00<00:00, 6078.70it/s]
